Finds the lag between the teensy log file and the 2p tif file using the aux barcode signal 

In [1]:
import tifffile
import numpy as np
import pandas as pd
import pickle
import scipy.io
from scipy.io import savemat
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal


import glob



import pynapple as nap
from pathlib import Path
from tqdm.notebook import tqdm
import scipy.signal as signal
import warnings
warnings.simplefilter('ignore')
warnings.filterwarnings("ignore", category=DeprecationWarning)



In [4]:

data_path = Path('/data/ofl_2p/A04_day2')

log_file = data_path.joinpath('20240502-113731_606_decoded.mat')
tif_file = data_path.joinpath('/data/ofl_2p/A04_day2/20240502_456225_00001.tif')
decoded_log = scipy.io.loadmat(str(log_file))
offset,crosscorr = auxSync(tif_file, decoded_log)

print (f'offset ={offset}')

0it [00:00, ?it/s]

offset =7.11299482304571


In [21]:
tif = tifffile.TiffFile(tif_file)

description = (tif.pages[0].tags.values()[5].value)

print(float(description.split('\n')[3].split('=')[-1]))

aux_line = description.split('\n')[10]
data = aux_line.split('=')[-1].strip(' [').strip(']').strip(' ]')
print(data)

0.0
-0.521224320 ,-0.521139485 ,-0.521071870 ,-0.521009185 ,-0.520772640 ,-0.520743360 ,-0.520714845 ,-0.520445695 ,-0.520186495 ,-0.520147775 ,-0.520110720 ,-0.520051295 ,-0.520039325 ,-0.511226275 ,-0.511143330 ,-0.511023395 ,-0.511020995 ,-0.511020485 ,-0.511019745 ,-0.510139650 ,-0.510126275 ,-0.510020870 ,-0.509953635 ,-0.509909445 ,-0.501236870 ,-0.501153065 ,-0.501086760 ,-0.501026695 ,-0.500972905 ,-0.500931910 ,-0.500860650 ,-0.500761480 ,-0.500733030 ,-0.500539945 ,-0.500466375 ,-0.500256905 ,-0.500192550 ,-0.500142695 ,-0.491245260 ,-0.491236490 ,-0.491094730 ,-0.491036910 ,-0.490985355 ,-0.490976525 ,-0.490835755 ,-0.490614540 ,-0.490204940 ,-0.490203500 ,-0.490199950 ,-0.490194380 ,-0.490156170 ,-0.490155115 ,-0.490150250 ,-0.490143210 ,-0.490106540 ,-0.490036330 ,-0.481257775 ,-0.481170290 ,-0.481101520 ,-0.481038930 ,-0.480983635 ,-0.480940690 ,-0.480868880 ,-0.480767505 ,-0.480682065 ,-0.480631955 ,-0.480465235 ,-0.480449040 ,-0.480443055 ,-0.480412015 ,-0.480268145 ,-0

In [ ]:
tag_structure = {'image_description': 5,
                 'frame_timestamp': 3,
                 'auxTrigger0': 10}  # position of aux tag in tif header

aux_data = {'frame_n': [],'ts': [], 'value': [] }

#reading the aux line from the tiff pages
with tifffile.TiffFile(tif_file) as tif:
    for i, page in tqdm(enumerate(tif.pages)):

        # extract image description string
        description = page.tags.values()[tag_structure['image_description']].value
        timestamp = float(description.split('\n')[tag_structure['frame_timestamp']].split('=')[-1])  # fetch timestamp in image description
        aux_line = description.split('\n')[tag_structure['auxTrigger0']]
        data = aux_line.split('=')[-1].strip(' [').strip(']').strip(' ]')

        aux_data['ts'].append(timestamp)
        aux_data['frame_n'].append(i)
        if len(data) > 0:
            aux_data['value'].append(1)


        else:

            aux_data['value'].append(0)
#panda format just bc looks nicer
aux_df = pd.DataFrame.from_dict(aux_data)


In [2]:

#logdata
digital_in = decoded_log['digitalIn'].astype(int)
digital_out = decoded_log['digitalOut'].astype(int)

#%timestamps of tiff aux signal
frame_ts = np.array(aux_df['ts']) #scanner clock
aux_tiff = np.array(aux_df['value'])
aux_tiff_idx = np.where(aux_tiff==1)[0] #indeces of aux signals
aux_tiff_ts = frame_ts[aux_tiff_idx] #timestamps for aux signal

#% log aux signal
log_ts = decoded_log['startTS'][0]
aux_log = digital_out[:,4] #channel for aux signal
aux_log_idx = np.where(np.diff(aux_log.astype(int)) == 1)[0] #indeces of when the signal is received
aux_log_ts = log_ts[aux_log_idx]/pow(10,6) #timestamps in seconds

aux_tiff_idx = np.where(aux_tiff==1)[0] # indeces of aux signals
aux_tiff_ts = frame_ts[aux_tiff_idx] # timestamps for 2p aux signal

#behavioural data logger

####### get rid off all the 1s except the middle one for each event
arr = aux_log
arr = np.insert(arr, 0, 0)
arr = np.append(arr, 0)
# Find the indices where the array changes
change_indices = np.where(np.diff(arr) != 0)[0]

# Create a new array of zeros with the same length as the original array
new_arr = np.zeros(arr.shape[0]-2, dtype=int)

# Iterate over the change indices in steps of 2 (start and end of each group of ones)
for i in range(0, len(change_indices), 2):
    # Calculate the middle index of the group of ones
    middle_index = (change_indices[i] + change_indices[i+1]) // 2 - 1
    # Set the middle index to 1 in the new array
    new_arr[middle_index] = 1

#########
#find the point of the dip as it has the largest diff and then go 1 further to exclude it
start = np.argmax(np.diff(log_ts[0:200000])) +1
log_ts_start = log_ts[start:]
aux_log_start = aux_log[start:]
new_arr_start = new_arr[start:]
##########

# create timestamps based on the data with the portion before start excluded
aux_log_idx_start =  np.where(new_arr_start==1)[0]# indeces of the signal onset (signal lasts more than 1ms)
aux_log_ts_start = log_ts_start[aux_log_idx_start]/pow(10,6) #timestamps in seconds

sr = 1/np.diff(frame_ts).mean()

bin_size =1/sr# in seconds (i.e. in the same unit as the aux_tiff_ts)
# doesnt matter too much, I guess smaller bin size is more accurate but as long as both same its ok

#create edges for binning
edges_2p = np.arange(0, aux_tiff_ts.max() + bin_size, bin_size)
edges_beh = np.arange(0, aux_log_ts_start.max() + bin_size, bin_size) #important to use the portion that has the start removed

#bin
hist_2p, bins = np.histogram(aux_tiff_ts, bins=edges_2p)
hist_beh, bins = np.histogram(aux_log_ts_start, bins=edges_beh)

#https://stackoverflow.com/questions/41492882/find-time-shift-of-two-signals-using-cross-correlation
def lag_finder(y2, y1, sr):
    n = min([len(y1),len(y2)])

    y1 = y1[0:n]
    y2 = y2[0:n]

    corr = signal.correlate(y2, y1, mode='same') / np.sqrt(signal.correlate(y1, y1, mode='same')[int(n/2)] *
                                                           signal.correlate(y2, y2, mode='same')[int(n/2)])
    offset_arr = np.linspace(-0.5*n/sr, 0.5*n/sr, n)
    offset = offset_arr[np.argmax(corr)]
    return offset,corr

offset,crosscorr = lag_finder(hist_2p, hist_beh[0:len(hist_2p)], sr)

return offset, crosscorr
